# Build Phase 1: Case 1

This notebook is the experiment entry point for the first Case 1 path. It shows the config, generates deterministic data, runs small non-GPU checks, documents the GPU kickoff command, and reads the output artifacts back into tables.


In [1]:
from pathlib import Path
import json
import torch
import yaml

from src.data.generate import generate_case1
from src.data.io import read_jsonl, filter_split
from src.metrics import exact_match, token_f1, format_validity, RetentionScores
from src.lanczos import l_lanczos

smoke_config_path = Path('configs/case1.yaml')
sweep_config_path = Path('configs/case1_small_sweep.yaml')
config = yaml.safe_load(smoke_config_path.read_text())
sweep_config = yaml.safe_load(sweep_config_path.read_text())

{
    'smoke_config_path': str(smoke_config_path),
    'sweep_config_path': str(sweep_config_path),
    'model': sweep_config['model']['name'],
    'task_a_path': sweep_config['data']['task_a_path'],
    'task_b_path': sweep_config['data']['task_b_path'],
    'metrics_path': sweep_config['outputs']['metrics_path'],
    'run_dir': sweep_config['outputs']['run_dir'],
}


/home/evan/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'model': {'name': 'microsoft/Phi-4-mini-instruct',
  'cache_dir': 'data/models',
  'load_in_4bit': True,
  'gradient_checkpointing': True,
  'attn_implementation': 'flash_attention_2',
  'trust_remote_code': False},
 'data': {'generated_dir': 'data/generated',
  'task_a_path': 'data/generated/case1_task_a.jsonl',
  'task_b_path': 'data/generated/case1_task_b.jsonl',
  'original_eval_path': 'data/generated/eval_original.jsonl',
  'n_per_split': 128,
  'seed': 13},
 'training': {'max_seq_length': 512,
  'per_device_batch_size': 1,
  'gradient_accumulation_steps': 8,
  'learning_rate': 0.0002,
  'epochs': 2,
  'qlora_rank': 8,
  'qlora_alpha': 16,
  'qlora_dropout': 0.05},
 'ewc': {'n0_values': [0, 4, 8, 16, 32, 64],
  'rank_values': [0, 1, 2, 4],
  'lambda_values': [0.1, 1.0, 10.0],
  'min_off_diag': 1e-12},
 'outputs': {'run_dir': 'outputs/case1',
  'metrics_path': 'outputs/case1/metrics.jsonl'}}

## Experiment Config

The smoke config is used for one-off notebook calls. The compact sweep config is what generated the artifacts analyzed below. The command is intentionally shown here so the `outputs/` directory is not mysterious.

In [ ]:
print(sweep_config_path.read_text())

In [ ]:
kickoff_command = (
    f"~/.venv/bin/python -m src.run_case1 "
    f"--config {sweep_config_path} "
    f"--limit-eval 8 "
    f"--resume"
)
print(kickoff_command)

## Generate Synthetic Data

In [2]:
generate_case1(
    output_dir=Path(config['data']['generated_dir']),
    seed=config['data']['seed'],
    n_per_split=8,
)

task_a = read_jsonl(config['data']['task_a_path'])
task_b = read_jsonl(config['data']['task_b_path'])
len(task_a), len(task_b)

(32, 32)

In [3]:
for example in task_a[:3]:
    print(json.dumps(example.__dict__, indent=2))

{
  "id": "task_a_ewc_init_000000",
  "task_family": "rule_transform",
  "task_id": "task_a",
  "split": "ewc_init",
  "prompt": "Routing transform: Convert product code EF-37 to its route label. Answer with only the route label.",
  "target": "ROUTE_BLUE_37",
  "answer_key": "ROUTE_BLUE_37",
  "metadata": {
    "entity_id": "EF-37",
    "requires_reasoning": false,
    "rule_id": "route_EF",
    "validator": {
      "pattern": "ROUTE_[A-Z]+_[0-9]{2}",
      "type": "regex"
    }
  }
}
{
  "id": "task_a_ewc_init_000001",
  "task_family": "rule_transform",
  "task_id": "task_a",
  "split": "ewc_init",
  "prompt": "Routing transform: Convert product code LM-87 to its route label. Answer with only the route label.",
  "target": "ROUTE_GREEN_87",
  "answer_key": "ROUTE_GREEN_87",
  "metadata": {
    "entity_id": "LM-87",
    "requires_reasoning": false,
    "rule_id": "route_LM",
    "validator": {
      "pattern": "ROUTE_[A-Z]+_[0-9]{2}",
      "type": "regex"
    }
  }
}
{
  "id": "task_

## Metric Smoke Checks

In [4]:
print('exact_match:', exact_match(' ROUTE_RED_01 ', 'route_red_01'))
print('token_f1:', token_f1('alpha beta gamma', 'alpha gamma delta'))
print('format_validity:', format_validity('ROUTE_RED_01', {'type': 'regex', 'pattern': r'ROUTE_[A-Z]+_[0-9]{2}'}))

retention = RetentionScores(score_before=0.8, score_after=0.6)
print('forgetting_delta:', retention.forgetting_delta)
print('retention_ratio:', retention.retention_ratio)

exact_match: 1.0
token_f1: 0.6666666666666666
format_validity: 1.0
forgetting_delta: 0.20000000000000007
retention_ratio: 0.7499999999999999


## Lanczos Smoke Check

In [5]:
grads = [torch.tensor([1.0, 0.0]), torch.tensor([0.0, 2.0])]

def get_grad_generator():
    return lambda: iter(grads)

L, diagonal = l_lanczos(
    get_grad_generator=get_grad_generator,
    r=2,
    p=2,
    calc_diag=True,
    disable_tqdm=True,
)
L.shape, diagonal.shape, torch.isfinite(L).all().item(), torch.isfinite(diagonal).all().item()

(torch.Size([2, 2]), torch.Size([2, 1]), True, True)

## Optional: Build Original-Model Retention Set

This downloads and runs `microsoft/Phi-4-mini-instruct`. Run after dependencies are installed and the GPU is available.

In [6]:
from src.modeling import ModelConfig, load_base_model, load_tokenizer
from src.original_retention import build_original_retention_set

tokenizer = load_tokenizer(config['model']['name'], config['model']['cache_dir'])
model = load_base_model(ModelConfig(**config['model']))
records = build_original_retention_set(model, tokenizer, Path(config['data']['original_eval_path']), repeats=3)
len(records)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
Loading weights: 100%|██████████| 194/194 [00:01<00:00, 192.59it/s, Materializing param=model.norm.weight]                              


8

## Optional: Run One Case 1 Smoke Training Job

This is GPU-heavy. Start with one small combination and a tiny eval limit before running the compact sweep command above. This call writes into the smoke config's `outputs/case1/` path.


In [7]:
from src.run_case1 import run_single

record = run_single(
    config=config,
    ewc_n0=4,
    ewc_rank=1,
    ewc_lambda=1.0,
    limit_eval=8,
)
record

Loading weights: 100%|██████████| 194/194 [00:01<00:00, 193.79it/s, Materializing param=model.norm.weight]                              
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/evan/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'case': 'case1',
 'train_n': 8,
 'ewc_n0': 4,
 'er_buffer': 0,
 'qlora_rank': 8,
 'ewc_rank': 1,
 'ewc_effective_rank': 1,
 'ewc_lambda': 1.0,
 'target_em': 0.0,
 'target_token_f1': 0.0,
 'target_format_validity': 0.0,
 'task_a_before_em': 0.0,
 'task_a_after_em': 0.0,
 'forgetting_delta': 0.0,
 'retention_ratio': 0.0,
 'average_retention': 0.0625,
 'worst_task_retention': 0.0,
 'eval_original_em': 0.125,
 'peak_vram_mb': 9935.08203125,
 'fisher_wall_seconds': 1.7080447673797607,
 'train_a_wall_seconds': 3.435971260070801,
 'train_b_wall_seconds': 3.3524091243743896,
 'run_wall_seconds': 57.40134000778198}

## Compact Sweep Artifacts

The cells below read the saved JSONL artifacts from the compact sweep config. Metrics come from `metrics.jsonl`; per-example predictions come from `predictions_*.jsonl`. The current sweep uses `limit_eval=8`, so treat retention ratios as noisy until we rerun on a larger eval slice.


In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

metrics_path = Path(sweep_config["outputs"]["metrics_path"])
run_dir = Path(sweep_config["outputs"]["run_dir"])
rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
df = pd.DataFrame(rows).sort_values(["ewc_n0", "ewc_rank", "ewc_lambda"]).reset_index(drop=True)

score_cols = [
    "ewc_n0",
    "ewc_rank",
    "ewc_effective_rank",
    "ewc_lambda",
    "target_em",
    "task_a_before_em",
    "task_a_after_em",
    "forgetting_delta",
    "retention_ratio",
    "eval_original_em",
    "average_retention",
    "worst_task_retention",
]

df[score_cols]


### Conflict-Aware Retention

Task A and Task B intentionally conflict only on the `AB`, `EF`, and `JK` routing rules. Task A maps those prefixes to `BLUE`; Task B maps them to `RED`. The `CD`, `GH`, and `LM` rules stay `GREEN` in both tasks. The aggregate Task A score can therefore split into conflicting and non-conflicting retention.

In [ ]:
import re

conflicting_rules = {"route_AB", "route_EF", "route_JK"}
task_a_examples = read_jsonl(sweep_config["data"]["task_a_path"])
metadata_by_id = {example.id: example.metadata for example in task_a_examples}

def prediction_path(split_name: str, ewc_n0: int, ewc_rank: int, ewc_lambda: float = 1.0) -> Path:
    return run_dir / f"predictions_{split_name}_n0-{ewc_n0}_rank-{ewc_rank}_lambda-{ewc_lambda:g}.jsonl"

def read_predictions(split_name: str, ewc_n0: int, ewc_rank: int, ewc_lambda: float = 1.0) -> pd.DataFrame:
    path = prediction_path(split_name, ewc_n0, ewc_rank, ewc_lambda)
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    pred_df = pd.DataFrame(rows)
    pred_df["rule_id"] = pred_df["id"].map(lambda row_id: metadata_by_id[row_id]["rule_id"])
    pred_df["conflict_group"] = pred_df["rule_id"].map(
        lambda rule_id: "conflicting" if rule_id in conflicting_rules else "nonconflicting"
    )
    return pred_df

conflict_records = []
for row in df.itertuples(index=False):
    for split_name in ["task_a_before", "task_a_after"]:
        pred_df = read_predictions(split_name, row.ewc_n0, row.ewc_rank, row.ewc_lambda)
        grouped = pred_df.groupby("conflict_group")["exact_match"].agg(["mean", "count"]).reset_index()
        for group_row in grouped.itertuples(index=False):
            conflict_records.append(
                {
                    "ewc_n0": row.ewc_n0,
                    "ewc_rank": row.ewc_rank,
                    "ewc_lambda": row.ewc_lambda,
                    "split": split_name,
                    "conflict_group": group_row.conflict_group,
                    "em": group_row.mean,
                    "n": group_row.count,
                }
            )

conflict_df = pd.DataFrame(conflict_records)
conflict_df.pivot_table(
    index=["ewc_n0", "ewc_rank"],
    columns=["split", "conflict_group"],
    values="em",
)

In [ ]:
example_split = read_predictions("task_a_after", ewc_n0=64, ewc_rank=4)
example_split[["id", "rule_id", "conflict_group", "target", "parsed_prediction", "exact_match"]]

### Read This First

For this tiny sweep, `target_em` is saturated at 1.0 for every row. The consistent `task_a_after_em = 0.5` is structural: the model keeps the non-conflicting GREEN rules and overwrites the conflicting BLUE rules with Task B's RED mapping. This is useful, but it means the aggregate Task A score should be read together with the conflict-aware table above.


In [ ]:
pivot_metrics = [
    "task_a_before_em",
    "task_a_after_em",
    "forgetting_delta",
    "retention_ratio",
    "eval_original_em",
    "fisher_wall_seconds",
    "run_wall_seconds",
]

for metric in pivot_metrics:
    print(f"\n{metric}")
    display(df.pivot(index="ewc_n0", columns="ewc_rank", values=metric))

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.copy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

for rank, group in plot_df.groupby("ewc_rank"):
    group = group.sort_values("ewc_n0")
    axes[0].plot(group["ewc_n0"], group["task_a_after_em"], marker="o", label=f"rank {rank}")
    axes[1].plot(group["ewc_n0"], group["retention_ratio"], marker="o", label=f"rank {rank}")
    axes[2].plot(group["ewc_n0"], group["fisher_wall_seconds"], marker="o", label=f"rank {rank}")

axes[0].set_title("Task A After EM")
axes[1].set_title("Retention Ratio")
axes[2].set_title("Fisher Seconds")
for ax in axes:
    ax.set_xlabel("EWC n0")
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel("score")
axes[1].set_ylabel("score")
axes[2].set_ylabel("seconds")
axes[0].legend()
plt.show()

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "best_task_a_after_em": [df["task_a_after_em"].max()],
        "best_retention_ratio": [df["retention_ratio"].max()],
        "all_target_em_saturated": [bool((df["target_em"] == 1.0).all())],
        "max_peak_vram_mb": [df["peak_vram_mb"].max()],
        "max_fisher_wall_seconds": [df["fisher_wall_seconds"].max()],
        "max_run_wall_seconds": [df["run_wall_seconds"].max()],
    }
)
summary

### Prediction artifacts

The runner writes prediction JSONL files for each combination. Use these to inspect exactly which examples are forgotten or retained. The default below opens the highest-`n0`, highest-rank row because it is the most expensive EWC estimate in this compact sweep.

In [ ]:
pred_path = prediction_path("task_a_after", ewc_n0=64, ewc_rank=4)
pred_rows = [json.loads(line) for line in pred_path.read_text().splitlines() if line.strip()]
preds = pd.DataFrame(pred_rows)
preds["rule_id"] = preds["id"].map(lambda row_id: metadata_by_id[row_id]["rule_id"])
preds["conflict_group"] = preds["rule_id"].map(lambda rule_id: "conflicting" if rule_id in conflicting_rules else "nonconflicting")

preds[["id", "rule_id", "conflict_group", "target", "parsed_prediction", "exact_match"]]


In [ ]:
# Change these knobs to inspect another row.
inspect_n0 = 4
inspect_rank = 4
inspect_split = "task_a_after"

inspect_path = prediction_path(inspect_split, inspect_n0, inspect_rank)
inspect_rows = [json.loads(line) for line in inspect_path.read_text().splitlines() if line.strip()]
inspect_df = pd.DataFrame(inspect_rows)

inspect_df[["prompt", "target", "parsed_prediction", "prediction", "exact_match"]].head(8)

### Current Interpretation

On this compact run, EWC does not yet show a clear retention win. Task B learning is solved in every row. Task A after-EM stays at 0.5 because the non-conflicting rules remain correct and the conflicting rules are overwritten. The useful artifact is that the machinery now exposes the tradeoff: increasing `EWC n0` and rank raises Fisher cost, but this tiny evaluation slice is too coarse to justify claiming a retention benefit. The next sharper run should keep the conflict-aware metrics, increase the eval slice, and probably repeat seeds before expanding lambda.
